[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/bigdata-getting-started/blob/main/notebooks/03_search.ipynb)

# 03 · Search — find the most relevant content

The **[Bigdata.com](https://bigdata.com)** Search Service (`POST /v1/search`)
retrieves the most relevant passages ("chunks") from a premium corpus of news,
earnings-call transcripts, filings, and investment research — plus your own
uploaded content — with optional live web results.

**What it's good for**
- Powering RAG pipelines and agents with accurate, sourced, real-time context
- Monitoring companies/themes across thousands of trusted sources
- Due diligence and event research with precise entity, topic and date filters

**Two modes**
- **`fast`** — runs a single query with exactly the filters you specify. Best
  when you control the filters (programmatic pipelines).
- **`smart`** — analyses the query text, auto-derives filters, and runs several
  sub-queries for better coverage. Best for raw user questions. (In smart mode
  only `timestamp` and `source` filters are allowed.)

In [1]:
!pip install -q requests env-colab-pass


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import requests
from env_colab_pass import passutil

# Your API key is read from the BIGDATA_API_KEY environment variable.
# Never hard-code keys in a notebook you plan to share or commit.
API_KEY = passutil.get_secret_value("BIGDATA_API_KEY")

# Bigdata exposes two hosts:
#   - api.bigdata.com    -> Knowledge Graph & Search Service (deterministic APIs)
#   - agents.bigdata.com -> Research Agent & Workflows (AI agents, streamed)
API_BASE = "https://api.bigdata.com"
AGENTS_BASE = "https://agents.bigdata.com"

# The key must be sent in the X-API-KEY header on every request.
HEADERS = {"X-API-KEY": API_KEY, "Content-Type": "application/json"}

A small helper so the examples stay readable. It prints the top chunks and the units consumed.

In [3]:
def search(query_text=None, mode="fast", max_chunks=5, show=3, **filters):
    """Call POST /v1/search. Pass filters=<dict> via the `filters` kwarg."""
    query = {"max_chunks": max_chunks}
    if query_text is not None:
        query["text"] = query_text
    if filters.get("filters"):
        query["filters"] = filters["filters"]
    payload = {"search_mode": mode, "query": query}
    r = requests.post(f"{API_BASE}/v1/search", headers=HEADERS, json=payload, timeout=90)
    r.raise_for_status()
    data = r.json()
    print(f"{len(data['results'])} documents · {data['usage']['api_query_units']} query units")
    for doc in data["results"][:show]:
        ch = doc["chunks"][0]
        print(f"\n\u2022 {doc['headline']}  [{doc['source']['name']}, {doc['timestamp'][:10]}]")
        print(f"  sentiment={ch['sentiment']:+.2f}  {ch['text'][:200]}...")
    return data

### Example 1 — Semantic search (fast)
Just describe what you want in natural language.

In [4]:
_ = search("government debt to GDP ratio outlook", max_chunks=5)

4 documents · 0.5 query units

• How bond markets can learn to love public debt  [Financial Times, 2026-06-11]
  sentiment=-0.14  To fix ideas, think about that huge potential white elephant that is Britain's HS2 high-speed train project. It may now cost about £100bn. That amounts to about 3.5 per cent of 2025 UK GDP. But suppos...

• Kraemer's Plain Talk: The Long End of the Yield Curve Takes Off - Public Finances Under Scrutiny  [Landesbank Baden Württemberg FI, 2026-06-12]
  sentiment=-0.24  Long-Term Bonds Under Pressure
Fig. 1: Industrialized Countries: Government Debt (% of GDP)
[Line chart showing the State debt of industrialised countries as a percentage of GDP from 2001 to 2031. The...

• IH Zimbabwe Consumer Sector Report 2026  [IH Securities, 2026-06-11]
  sentiment=-0.59  - According to the World Bank, the region's general government debt reached US$1.26tn in 2025, a record in absolute terms, yet the debt-to-GDP ratio declined for a second consecutive year, a reflectio...


### Example 2 — Filter by entity
Target a specific company with its Knowledge Graph entity ID (see notebook 02).
`any_of` matches chunks mentioning any of the listed entities.

In [5]:
# Resolve Apple's entity ID on the fly.
apple = requests.post(f"{API_BASE}/v1/knowledge-graph/companies", headers=HEADERS,
                      json={"query": "Apple", "types": ["PUBLIC"]}, timeout=30).json()["results"][0]["id"]

_ = search("supply chain and manufacturing", max_chunks=5,
           filters={"entity": {"any_of": [apple]}})

5 documents · 0.5 query units

• Fund Scale Surges, Reaching New Highs  [Chinatimes, 2026-06-14]
  sentiment=+0.64  Nvidia has doubled its supply volume for the second half of this year, and Vera Rubin has entered mass production, with industry scale expected to expand further by 2027. New chips will drive upgrades...

• America's Rare Earth Reckoning Could Create a New Strategic Powerhouse  [Yahoo! Finance, 2026-06-04]
  sentiment=+0.51  The importance of that supply chain extends well beyond defense. Aerospace manufacturers such as GE Aerospace (NYSE:GE) rely on rare earth magnets and advanced materials across jet engines, avionics, ...

• Glass Substrates Break Through AI Packaging Limits, Addressing Mass Production Bottlenecks is the True Supply Chain  [Chinatimes, 2026-06-14]
  sentiment=-0.22  However, the risks must be stated upfront. CoPoS is still in the critical stage of validation from pilot production to mass production. Glass substrates, TGV, warpage control, RDL uniformit

### Example 3 — Filter by keyword
Require specific terms to appear. `all_of` / `any_of` / `none_of` are supported,
and you can restrict to `HEADLINE`, `BODY`, or `ALL`.

In [6]:
_ = search("financial institutions sanctions compliance obligations", max_chunks=5,
           filters={"keyword": {"any_of": ["Sanctions", "Compliance"]}})

5 documents · 0.5 query units

• Navan, Inc. files FORM 10-Q for Q1, FY 2027 on Jun 11, 2026  [Edgar SEC Filings, 2026-06-11]
  sentiment=-0.69  Our financial institution partners as well as regulators in the United States and globally continue to increase their scrutiny of compliance with these obligations, which may require us to further inv...

• First Carolina Financial Services Inc: Conference Call on Jun 12, 2026 - Report  [Quartr Reports, 2026-06-12]
  sentiment=-0.72  Banking regulators examine banks for compliance with the BSA, USA PATRIOT Act, and economic sanctions regulations administered by OFAC, and failure of a financial institution to maintain and implement...

• Narragansett Bancorp, Inc. files FORM S-1 on Jun 12, 2026  [Edgar SEC Filings, 2026-06-12]
  sentiment=-0.58  Non-compliance with the USA PATRIOT Act, Bank Secrecy Act, or other laws and regulations could result in fines or sanctions.
The USA PATRIOT and Bank Secrecy Acts require financial institutions to dev..

### Example 4 — Filter by sentiment
Every chunk has a market-impact sentiment score from -1.0 to 1.0. Filter by
explicit ranges (here: clearly negative).

In [7]:
_ = search("climate transition risk for banks", max_chunks=5,
           filters={"sentiment": {"ranges": [{"min": -1.0, "max": -0.1}]}})

5 documents · 0.5 query units

• Green Fintech 2026 - Unlocking Value Across The Green Fintech Revolution  [CrispIdea, 2026-06-11]
  sentiment=-0.62  Risks & Challenges
- Regulatory Fragmentation & Compliance Fatigue
As the global transition to sustainable finance intensifies, Regulatory Fragmentation has emerged as a top operational risk for multi...

• Itau Unibanco Holding S.A. files FORM 6-K on Jun 11, 2026  [Edgar SEC Filings, 2026-06-11]
  sentiment=-0.78  Extreme climate events can weaken the ability of individuals and businesses in impacted areas to meet their financial obligations, resulting in higher default rates. Moreover, these disruptions may co...

• Doha Bank QSC - Doha Finance Limited and Doha Bank Q.P.S.C. - Base Offering Circular  [PubT, 2026-05-15]
  sentiment=-0.84  The Bank cannot ensure that it will be able to adequately address these concerns, which could prevent the Bank from achieving its strategic objectives and could also adversely affect the Bank's busine..

### Example 5 — Filter by date range
Restrict to a publication window (ISO 8601, UTC).

In [8]:
_ = search("interest rate policy outlook", max_chunks=5,
           filters={"timestamp": {"start": "2025-01-01T00:00:00Z", "end": "2025-12-31T23:59:59Z"}})

5 documents · 0.5 query units

• Fed expected to cut rates despite deep divisions over US economic outlook  [Financial Times, 2025-12-07]
  sentiment=-0.13  The Federal Reserve is set to cut interest rates next week despite deep divisions among its officials on the direction of the US economy, according to leading academic economists.
The rate-setting Fed...

• Pound Sterling flattens against US Dollar as markets turn to FOMC Minutes  [FXStreet News, 2025-12-30]
  sentiment=-0.07  Investors await the FOMC minutes to get detailed cues on policymakers' views on the monetary policy outlook. In the last policy meeting, the Fed delivered its third interest rate cut of year, pushing ...

• US Equity Indexes Decline as Rising Treasury Yields Signal Divided Fed, Cautious Interest Rates Outlook  [MT Newswires, 2025-12-08]
  sentiment=-0.35  In the case of the latter, the door will remain open to further cuts in early 2026.
The current probability of the Fed announcing a 25-basis-point reduction

### Example 6 — Filter by document type
Narrow to a content type, e.g. only earnings-call transcripts.

In [9]:
_ = search("management guidance for next quarter", max_chunks=5,
           filters={"document_type": {"mode": "INCLUDE", "values": [{"type": "TRANSCRIPT"}]}})

5 documents · 0.5 query units

• Adyen NV: Annual General Meeting  [Factset Transcripts, 2026-05-28]
  sentiment=-0.06  So in the past, we've worked with the guidance across multiple years and I think the consistent feedback from investors was not helping to get better clarity on individual years. So that's exactly why...

• Medline Inc: Conference on Jun 9, 2026 - Transcript  [Quartr Transcripts, 2026-06-09]
  sentiment=+0.16  Before I want to go to margins, just open up to see if there are any questions from people in the audience. Okay, we'll keep going. Before I actually get to margins, maybe we just kind of, it's kind o...

• "Alexa, Let's Go to Outer Space"  [Motley Fool Money, 2026-04-02]
  sentiment=-0.49  But if you look ahead to the first quarter, it's looking at a sales decline. And this is after already a couple of years of just kind of mediocre lackluster results. But then guidance for the year is ...


### Example 7 — Smart search
Send a raw question and let Bigdata.com derive the filters and run sub-queries.

In [10]:
_ = search("How is Nvidia exposed to US-China export restrictions?", mode="smart", max_chunks=8)

7 documents · 0.8 query units

• Global Artificial Intelligence Industry June 2026  [GSBR Research Private Limited, 2026-06-09]
  sentiment=+0.36  Recent shifts in US export policy have softened prior restrictions on advanced Al chips, allowing NVIDIA to seek approval to export its high- performance H200 Al processors to China under a new case-b...

• Nvidia Corp. - An $80 Billion Buyback  [CrispIdea, 2026-06-10]
  sentiment=-0.72  Risks
- Dependence on Overseas Manufacturing: The company relies completely on overseas manufacturing partners, mainly in Taiwan and South Korea, which exposes it to severe supply disruptions.
- Vulne...

• NVIDIA CORP files FORM 10-Q for Q1, FY 2027 on May 20, 2026  [Edgar SEC Filings, 2026-05-20]
  sentiment=-0.85  In the event that we are able to sell licensed products into the China market, we may not be able to pass along all or any of the tariff to our customers, and may be subject to litigation, increased c...


## Fast vs. Smart — quick guide
| | fast | smart |
|---|---|---|
| Filters | all filters, exactly as given | only `timestamp` & `source` (others auto-derived) |
| Sub-queries | one | several, for coverage |
| Best for | controlled pipelines | Invoked by Agent and raw user questions |

## Batch Search
Need to run **many** queries efficiently (e.g. a whole watchlist or a screen)?
Use the **Batch Search API** (`/v1/search/batches`). See the cookbook:
👉 https://github.com/Bigdata-com/bigdata-cookbook/tree/main/Batch_Search_API

**Next:** [`04_research.ipynb`](04_research.ipynb)

_Source: [Bigdata.com](https://bigdata.com) · Search docs: https://docs.bigdata.com/api-reference/search/search-documents_